# 🎙️ Genuine Kyutai Moshi Server on Free Kaggle GPU
### Research Backend for Adaptive, Accessibility-Aware Voice Agent Harness

---

## ⚡ Pre-Flight Verification Checklist
1. **Phone Verification**: Check that your Kaggle account has completed phone verification at [kaggle.com/settings](https://www.kaggle.com/settings) (required to turn Internet ON).
2. **Right Sidebar Settings (Notebook Options)**:
   - **Accelerator**: Select **`GPU T4 x2`** (Recommended: 2x 16GB GPUs, 30 free hours/week) or **`GPU P100`**.
   - **Internet**: Toggle to **`ON`** (required to download PyTorch weights & open tunnel).
3. **Execution**: Run the cells in order. The server will download the genuine `kyutai/moshiko-pytorch-bf16` weights and display the live public Gradio URL.

In [ ]:
# ==============================================================================
# STAGE 1: Environment Diagnostics & System Verification
# ==============================================================================
import sys
import os
import shutil
import psutil
import torch

print("=" * 70)
print("📋 STAGE 1: ENVIRONMENT & RESOURCE AUDIT")
print("=" * 70)
print(f"• Python Version   : {sys.version.split()[0]}")
print(f"• PyTorch Version  : {torch.__version__}")
print(f"• CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"• CUDA Version     : {torch.version.cuda}")
    print(f"• GPU Count        : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        dev_name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  - GPU {i}            : {dev_name} ({vram:.1f} GB VRAM)")
else:
    print("⚠️ WARNING: No GPU detected! Please enable 'GPU T4 x2' in the right sidebar.")

# Disk & RAM Check
disk = shutil.disk_usage("/kaggle/working")
ram = psutil.virtual_memory()
print(f"• Free Disk Space  : {disk.free / (1024**3):.1f} GB (Required: ~16 GB)")
print(f"• Available RAM    : {ram.available / (1024**3):.1f} GB")
print("=" * 70)

In [ ]:
# ==============================================================================
# STAGE 2: GPU Hardware Diagnostics (nvidia-smi)
# ==============================================================================
!nvidia-smi

In [ ]:
# ==============================================================================
# STAGE 3: Install Official Kyutai Moshi & Gradio Tunneling
# ==============================================================================
print("Installing Kyutai Moshi & Gradio dependencies...")
!pip install -q -U moshi gradio

# Verify import
import moshi
print(f"✅ Kyutai Moshi installed successfully (Version: {getattr(moshi, '__version__', 'latest')})")

In [ ]:
# ==============================================================================
# STAGE 4: Model Weight Feasibility Check
# Model: kyutai/moshiko-pytorch-bf16 (7B Helium backbone + Mimi neural audio codec)
# Download size: ~14.8 GB | VRAM Footprint: ~14.5 GB
# ==============================================================================
print("Estimating Model Requirements:")
print("  • Target Repo     : kyutai/moshiko-pytorch-bf16")
print("  • Precision       : bfloat16 (16-bit)")
print("  • Download Size   : ~14.8 GB")
print("  • Required VRAM   : ~14.5 GB (Fits comfortably in 16GB T4 / P100)")
print("  • Architecture    : 7B Multi-Stream Full-Duplex Speech-Text Transformer")
print("  • Audio Codec     : Mimi 24 kHz (12.5 Hz framerate, 1.1 kbps)")

In [ ]:
# ==============================================================================
# STAGE 5: Launch Genuine Kyutai Moshi Server with Memory Optimization
# Prevents CUDA OOM on 15GB Tesla T4 GPUs by avoiding memory fragmentation.
# ==============================================================================
import os
import gc
import sys
import torch

# 1. Configure PyTorch execution environment
# - NO_CUDA_GRAPH=1: Disables CUDA graph capture so multi-GPU layer transfers work without stream capture errors.
# - NO_TORCH_COMPILE=1: Disables JIT compile since Tesla T4 lacks native bfloat16 compile support.
# - PYTORCH_ALLOC_CONF: Prevents VRAM allocator fragmentation.
os.environ["NO_CUDA_GRAPH"] = "1"
os.environ["NO_TORCH_COMPILE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Multi-GPU Layer Distribution & Memory-Safe Loader
# Kaggle provides 2x Tesla T4 GPUs (GPU 0 & GPU 1, 30GB total). We balance the 7B model:
# - Layers 0..15 on GPU 0 (~7.5 GB)
# - Layers 16..31 on GPU 1 (~7.5 GB)
# - Mimi Audio Codec on GPU 1 (~0.3 GB)
try:
    import moshi.modules.transformer as tf_mod
    orig_tf_forward = tf_mod.StreamingTransformer.forward
    def multi_gpu_tf_forward(self, x, *args, **kwargs):
        target_device = x.device
        for layer in self.layers:
            layer_dev = next(layer.parameters()).device
            if x.device != layer_dev:
                x = x.to(layer_dev)
            x = layer(x, *args, **kwargs)
        return x.to(target_device)
    tf_mod.StreamingTransformer.forward = multi_gpu_tf_forward

    import moshi.models.loaders as loaders
    class CrossDeviceMimi:
        def __init__(self, mimi, mimi_device, target_device="cuda:0"):
            self._mimi = mimi
            self._mimi_device = torch.device(mimi_device)
            self._target_device = torch.device(target_device)
        def encode(self, chunk):
            chunk = chunk.to(self._mimi_device)
            codes = self._mimi.encode(chunk)
            return codes.to(self._target_device)
        def decode(self, tokens):
            tokens = tokens.to(self._mimi_device)
            pcm = self._mimi.decode(tokens)
            return pcm.to(self._target_device)
        def __getattr__(self, name):
            return getattr(self._mimi, name)

    orig_get_mimi = loaders.CheckpointInfo.get_mimi
    def multi_gpu_get_mimi(self, device="cuda"):
        mimi_dev = "cuda:1" if torch.cuda.device_count() > 1 else "cpu"
        print(f"[Multi-GPU] Offloading Mimi Audio Codec to {mimi_dev}...")
        raw_mimi = orig_get_mimi(self, device=mimi_dev)
        print(f"✅ Mimi loaded on {mimi_dev}!")
        return CrossDeviceMimi(raw_mimi, mimi_device=mimi_dev, target_device=device)
    loaders.CheckpointInfo.get_mimi = multi_gpu_get_mimi

    orig_load_file = loaders.load_file
    def safe_load_file(filename, device="cuda"):
        print("[Multi-GPU] Loading safetensors via CPU RAM and balancing layers across GPU 0 & GPU 1...")
        cpu_state = orig_load_file(filename, device="cpu")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        has_two_gpus = torch.cuda.device_count() > 1
        target_state = {}
        for k, v in cpu_state.items():
            target_dev = str(device)
            if has_two_gpus and k.startswith("transformer.layers."):
                try:
                    layer_idx = int(k.split(".")[2])
                    if layer_idx >= 16:
                        target_dev = "cuda:1"
                except Exception:
                    pass
            if v.dtype.is_floating_point:
                if k.startswith('condition_provider.') or k.startswith('fuser.'):
                    v = v.float()
                else:
                    v = v.to(torch.bfloat16)
            target_state[k] = v.to(target_dev)
        del cpu_state
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("✅ 7B Model balanced successfully: Layers 0-15 on GPU 0, Layers 16-31 on GPU 1!")
        return target_state
    loaders.load_file = safe_load_file
except Exception as e:
    print(f"Notice: {e}")

# 3. Set CLI arguments BEFORE importing moshi.server (moshi/server.py auto-executes main() on import)
print("=" * 70)
print("🚀 Launching Kyutai Moshi Full-Duplex Server...")
print("👉 Look for: 'Running on public URL: https://xxxxxxxxxxxx.gradio.live'")
print("👉 Copy that URL and paste it into your local Harness dashboard at http://localhost:8501")
print("=" * 70 + "\n")

sys.argv = [
    "server.py",
    "--gradio-tunnel",
    "--hf-repo", "kyutai/moshiko-pytorch-bf16"
]

# Importing moshi.server automatically triggers main() using our clean sys.argv
import moshi.server
